# EMNLP 2025 Research Gap Analysis — Step 5: Full-Text Validation

This notebook validates candidate research gaps from Cluster 7 using **full paper text** rather than abstracts alone.

Primary candidate directions:

1. Multilingual reliability of LLM-as-a-Judge
2. Robustness / stability of LLM-as-a-Judge

Workflow:

1. Load the 18 Cluster 7 papers
2. Download PDFs from `pdf_url`
3. Extract full text from PDFs
4. Search for relevant sections and keywords
5. Use an LLM to conservatively code the full-text evidence
6. Compare abstract-level findings against full-text findings
7. Produce a validation table for candidate research gaps

Expected input:
`../data/processed/cluster7_llm_judge_papers.csv`


## 0. Install additional dependencies if needed

Run this once if your environment does not already have these packages.


In [1]:
# Uncomment if needed:
%pip install requests pymupdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.9/23.9 MB 19.6 MB/s  0:00:010.5 MB/s eta 0:00:01:01
Note: you may need to restart the kernel to use updated packages.


## 1. Load libraries and Cluster 7 papers


In [2]:
import os
import re
import json
import time
import requests
import pandas as pd
import fitz  # PyMuPDF

from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

INPUT_FILE = Path("../data/processed/cluster7_llm_judge_papers.csv")
PDF_DIR = Path("../data/fulltext/cluster7")
TEXT_DIR = Path("../data/fulltext/cluster7_text")

PDF_DIR.mkdir(parents=True, exist_ok=True)
TEXT_DIR.mkdir(parents=True, exist_ok=True)

papers = pd.read_csv(INPUT_FILE)

print("Loaded papers:", len(papers))
papers[["paper_id", "title", "pdf_url"]].head()


Loaded papers: 18


,paper_id,title,pdf_url
0,2025.emnlp-main.86,Beyond the Surface: Measuring Self-Preference ...,https://aclanthology.org/2025.emnlp-main.86.pdf
1,2025.emnlp-main.103,Direct Judgement Preference Optimization,https://aclanthology.org/2025.emnlp-main.103.pdf
2,2025.emnlp-main.138,From Generation to Judgment: Opportunities and...,https://aclanthology.org/2025.emnlp-main.138.pdf
3,2025.emnlp-main.360,Arena-lite: Efficient and Reliable Large Langu...,https://aclanthology.org/2025.emnlp-main.360.pdf
4,2025.emnlp-main.538,Are Checklists Really Useful for Automatic Eva...,https://aclanthology.org/2025.emnlp-main.538.pdf


## 2. Download PDFs

The notebook skips files that already exist, so rerunning it will not redownload successful downloads.


In [3]:
def safe_filename(value):
    value = str(value)
    value = re.sub(r"[^A-Za-z0-9._-]+", "_", value)
    return value[:120]

def download_pdf(url, output_path, timeout=60):
    if output_path.exists() and output_path.stat().st_size > 1000:
        return "exists"

    if pd.isna(url) or not str(url).strip():
        return "missing_url"

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    try:
        response = requests.get(
            str(url),
            headers=headers,
            timeout=timeout,
            allow_redirects=True
        )
        response.raise_for_status()

        content_type = response.headers.get("content-type", "").lower()

        if "pdf" not in content_type and not response.content.startswith(b"%PDF"):
            return f"not_pdf ({content_type})"

        output_path.write_bytes(response.content)
        return "downloaded"

    except Exception as e:
        return f"error: {e}"


download_results = []

for i, row in papers.iterrows():
    pdf_path = PDF_DIR / f"{safe_filename(row['paper_id'])}.pdf"

    status = download_pdf(
        row.get("pdf_url", ""),
        pdf_path
    )

    download_results.append({
        "paper_id": row["paper_id"],
        "title": row["title"],
        "pdf_path": str(pdf_path),
        "download_status": status
    })

    print(f"{i+1:02d}/{len(papers)} | {status} | {row['title']}")

download_df = pd.DataFrame(download_results)

download_df


01/18 | downloaded | Beyond the Surface: Measuring Self-Preference in LLM Judgments
02/18 | downloaded | Direct Judgement Preference Optimization
03/18 | downloaded | From Generation to Judgment: Opportunities and Challenges of LLM -as-a-judge
04/18 | downloaded | Arena-lite: Efficient and Reliable Large Language Model Evaluation via Tournament-Based Direct Comparisons
05/18 | downloaded | Are Checklists Really Useful for Automatic Evaluation of Generative Tasks?
06/18 | downloaded | Analyzing Uncertainty of LLM -as-a-Judge: Interval Evaluations with Conformal Prediction
07/18 | downloaded | Meta-Rewarding Language Models: Self-Improving Alignment with LLM -as-a-Meta-Judge
08/18 | downloaded | Self-Augmented Preference Alignment for Sycophancy Reduction in LLM s
09/18 | downloaded | Improve LLM -as-a-Judge Ability as a General Ability
10/18 | downloaded | Towards Optimal Evaluation Efficiency for Large Language Models
11/18 | downloaded | C heck E val: A reliable LLM -as-a-Judge framew

,paper_id,title,pdf_path,download_status
0,2025.emnlp-main.86,Beyond the Surface: Measuring Self-Preference ...,../data/fulltext/cluster7/2025.emnlp-main.86.pdf,downloaded
1,2025.emnlp-main.103,Direct Judgement Preference Optimization,../data/fulltext/cluster7/2025.emnlp-main.103.pdf,downloaded
2,2025.emnlp-main.138,From Generation to Judgment: Opportunities and...,../data/fulltext/cluster7/2025.emnlp-main.138.pdf,downloaded
3,2025.emnlp-main.360,Arena-lite: Efficient and Reliable Large Langu...,../data/fulltext/cluster7/2025.emnlp-main.360.pdf,downloaded
4,2025.emnlp-main.538,Are Checklists Really Useful for Automatic Eva...,../data/fulltext/cluster7/2025.emnlp-main.538.pdf,downloaded
5,2025.emnlp-main.569,Analyzing Uncertainty of LLM -as-a-Judge: Inte...,../data/fulltext/cluster7/2025.emnlp-main.569.pdf,downloaded
6,2025.emnlp-main.583,Meta-Rewarding Language Models: Self-Improving...,../data/fulltext/cluster7/2025.emnlp-main.583.pdf,downloaded
7,2025.emnlp-main.625,Self-Augmented Preference Alignment for Sycoph...,../data/fulltext/cluster7/2025.emnlp-main.625.pdf,downloaded
8,2025.emnlp-main.712,Improve LLM -as-a-Judge Ability as a General A...,../data/fulltext/cluster7/2025.emnlp-main.712.pdf,downloaded
9,2025.emnlp-main.716,Towards Optimal Evaluation Efficiency for Larg...,../data/fulltext/cluster7/2025.emnlp-main.716.pdf,downloaded


## 3. Check download success


In [4]:
print(download_df["download_status"].value_counts())

failed_downloads = download_df[
    ~download_df["download_status"].isin(["downloaded", "exists"])
]

failed_downloads


download_status
downloaded    18
Name: count, dtype: int64


,paper_id,title,pdf_path,download_status


## 4. Extract full text from PDFs

PyMuPDF extracts text page-by-page. The raw extracted text is saved to disk so it can be reused without parsing the PDF again.


In [5]:
def extract_pdf_text(pdf_path):
    doc = fitz.open(pdf_path)

    pages = []

    for page_num, page in enumerate(doc, start=1):
        text = page.get_text("text")
        pages.append(
            f"\n\n===== PAGE {page_num} =====\n\n{text}"
        )

    doc.close()

    return "".join(pages)


text_results = []

for i, row in download_df.iterrows():
    if row["download_status"] not in ["downloaded", "exists"]:
        continue

    pdf_path = Path(row["pdf_path"])
    text_path = TEXT_DIR / f"{safe_filename(row['paper_id'])}.txt"

    try:
        if text_path.exists() and text_path.stat().st_size > 500:
            full_text = text_path.read_text(
                encoding="utf-8",
                errors="ignore"
            )
            status = "exists"
        else:
            full_text = extract_pdf_text(pdf_path)
            text_path.write_text(
                full_text,
                encoding="utf-8"
            )
            status = "extracted"

        text_results.append({
            "paper_id": row["paper_id"],
            "title": row["title"],
            "text_path": str(text_path),
            "characters": len(full_text),
            "text_status": status
        })

        print(
            f"{i+1:02d}/{len(download_df)} | "
            f"{status} | {len(full_text):,} chars | "
            f"{row['title']}"
        )

    except Exception as e:
        text_results.append({
            "paper_id": row["paper_id"],
            "title": row["title"],
            "text_path": str(text_path),
            "characters": 0,
            "text_status": f"error: {e}"
        })

        print("ERROR:", row["title"], e)

text_df = pd.DataFrame(text_results)

text_df


01/18 | extracted | 66,086 chars | Beyond the Surface: Measuring Self-Preference in LLM Judgments
02/18 | extracted | 128,455 chars | Direct Judgement Preference Optimization
03/18 | extracted | 160,555 chars | From Generation to Judgment: Opportunities and Challenges of LLM -as-a-judge
04/18 | extracted | 71,043 chars | Arena-lite: Efficient and Reliable Large Language Model Evaluation via Tournament-Based Direct Comparisons
05/18 | extracted | 92,541 chars | Are Checklists Really Useful for Automatic Evaluation of Generative Tasks?
06/18 | extracted | 136,916 chars | Analyzing Uncertainty of LLM -as-a-Judge: Interval Evaluations with Conformal Prediction
07/18 | extracted | 55,829 chars | Meta-Rewarding Language Models: Self-Improving Alignment with LLM -as-a-Meta-Judge
08/18 | extracted | 49,534 chars | Self-Augmented Preference Alignment for Sycophancy Reduction in LLM s
09/18 | extracted | 65,427 chars | Improve LLM -as-a-Judge Ability as a General Ability
10/18 | extracted | 22,6

,paper_id,title,text_path,characters,text_status
0,2025.emnlp-main.86,Beyond the Surface: Measuring Self-Preference ...,../data/fulltext/cluster7_text/2025.emnlp-main...,66086,extracted
1,2025.emnlp-main.103,Direct Judgement Preference Optimization,../data/fulltext/cluster7_text/2025.emnlp-main...,128455,extracted
2,2025.emnlp-main.138,From Generation to Judgment: Opportunities and...,../data/fulltext/cluster7_text/2025.emnlp-main...,160555,extracted
3,2025.emnlp-main.360,Arena-lite: Efficient and Reliable Large Langu...,../data/fulltext/cluster7_text/2025.emnlp-main...,71043,extracted
4,2025.emnlp-main.538,Are Checklists Really Useful for Automatic Eva...,../data/fulltext/cluster7_text/2025.emnlp-main...,92541,extracted
5,2025.emnlp-main.569,Analyzing Uncertainty of LLM -as-a-Judge: Inte...,../data/fulltext/cluster7_text/2025.emnlp-main...,136916,extracted
6,2025.emnlp-main.583,Meta-Rewarding Language Models: Self-Improving...,../data/fulltext/cluster7_text/2025.emnlp-main...,55829,extracted
7,2025.emnlp-main.625,Self-Augmented Preference Alignment for Sycoph...,../data/fulltext/cluster7_text/2025.emnlp-main...,49534,extracted
8,2025.emnlp-main.712,Improve LLM -as-a-Judge Ability as a General A...,../data/fulltext/cluster7_text/2025.emnlp-main...,65427,extracted
9,2025.emnlp-main.716,Towards Optimal Evaluation Efficiency for Larg...,../data/fulltext/cluster7_text/2025.emnlp-main...,22637,extracted


## 5. Keyword-based evidence scan

This is not the final gap analysis. It is a fast way to identify papers likely to contain evidence about:

- multilingual evaluation
- low-resource languages
- robustness / sensitivity
- prompt variation
- position or order bias
- human evaluation
- limitations and future work


In [6]:
KEYWORD_GROUPS = {
    "multilingual": [
        "multilingual",
        "cross-lingual",
        "cross lingual",
        "language",
        "languages",
        "low-resource",
        "low resource",
        "non-english",
        "non english"
    ],
    "robustness": [
        "robust",
        "robustness",
        "stability",
        "consistent",
        "consistency",
        "sensitivity",
        "perturb",
        "variation"
    ],
    "prompt_variation": [
        "prompt wording",
        "prompt variation",
        "prompt sensitivity",
        "instruction variation",
        "different prompts",
        "prompt template"
    ],
    "judge_bias": [
        "position bias",
        "order bias",
        "self-preference",
        "self preference",
        "verbosity bias",
        "length bias",
        "style bias",
        "bias"
    ],
    "human_comparison": [
        "human evaluation",
        "human evaluator",
        "human annotator",
        "human judgment",
        "human judgement",
        "human agreement"
    ],
    "limitations": [
        "limitations",
        "limitation"
    ],
    "future_work": [
        "future work",
        "future research",
        "future direction"
    ]
}

def keyword_counts(text, keywords):
    text_lower = text.lower()
    return sum(text_lower.count(keyword.lower()) for keyword in keywords)


keyword_rows = []

for _, row in text_df.iterrows():
    if not str(row["text_status"]).startswith(("extracted", "exists")):
        continue

    text = Path(row["text_path"]).read_text(
        encoding="utf-8",
        errors="ignore"
    )

    record = {
        "paper_id": row["paper_id"],
        "title": row["title"]
    }

    for group, keywords in KEYWORD_GROUPS.items():
        record[group] = keyword_counts(text, keywords)

    keyword_rows.append(record)

keyword_df = pd.DataFrame(keyword_rows)

keyword_df


,paper_id,title,multilingual,robustness,prompt_variation,judge_bias,human_comparison,limitations,future_work
0,2025.emnlp-main.86,Beyond the Surface: Measuring Self-Preference ...,20,7,0,233,1,4,0
1,2025.emnlp-main.103,Direct Judgement Preference Optimization,53,31,5,94,4,2,1
2,2025.emnlp-main.138,From Generation to Judgment: Opportunities and...,207,39,1,69,7,18,10
3,2025.emnlp-main.360,Arena-lite: Efficient and Reliable Large Langu...,12,31,0,13,4,4,2
4,2025.emnlp-main.538,Are Checklists Really Useful for Automatic Eva...,26,23,0,3,37,10,2
5,2025.emnlp-main.569,Analyzing Uncertainty of LLM -as-a-Judge: Inte...,30,52,1,20,4,7,0
6,2025.emnlp-main.583,Meta-Rewarding Language Models: Self-Improving...,23,6,0,17,1,4,0
7,2025.emnlp-main.625,Self-Augmented Preference Alignment for Sycoph...,28,11,6,29,0,4,3
8,2025.emnlp-main.712,Improve LLM -as-a-Judge Ability as a General A...,18,13,3,22,0,8,0
9,2025.emnlp-main.716,Towards Optimal Evaluation Efficiency for Larg...,12,3,0,0,0,4,2


## 6. Retrieve relevant evidence windows

Instead of sending entire PDFs to the LLM, this notebook extracts text windows around terms relevant to the two candidate gaps.

This makes the review cheaper and easier to audit.


In [7]:
def evidence_windows(text, keywords, window=1200, max_windows=12):
    text_lower = text.lower()
    hits = []

    for keyword in keywords:
        start = 0

        while True:
            pos = text_lower.find(keyword.lower(), start)

            if pos == -1:
                break

            left = max(0, pos - window)
            right = min(len(text), pos + len(keyword) + window)

            snippet = text[left:right].strip()

            hits.append({
                "keyword": keyword,
                "position": pos,
                "snippet": snippet
            })

            start = pos + len(keyword)

    # Deduplicate nearby hits
    hits = sorted(hits, key=lambda x: x["position"])

    deduped = []

    for hit in hits:
        if (
            not deduped
            or hit["position"] - deduped[-1]["position"] > window
        ):
            deduped.append(hit)

    return deduped[:max_windows]


ALL_GAP_KEYWORDS = sorted(set(
    KEYWORD_GROUPS["multilingual"]
    + KEYWORD_GROUPS["robustness"]
    + KEYWORD_GROUPS["prompt_variation"]
    + KEYWORD_GROUPS["judge_bias"]
    + KEYWORD_GROUPS["human_comparison"]
    + KEYWORD_GROUPS["limitations"]
    + KEYWORD_GROUPS["future_work"]
))


evidence_records = []

for _, row in text_df.iterrows():
    if not str(row["text_status"]).startswith(("extracted", "exists")):
        continue

    text = Path(row["text_path"]).read_text(
        encoding="utf-8",
        errors="ignore"
    )

    windows = evidence_windows(
        text,
        ALL_GAP_KEYWORDS,
        window=1200,
        max_windows=15
    )

    evidence_text = "\n\n--- EVIDENCE WINDOW ---\n\n".join(
        [
            f"Matched keyword: {w['keyword']}\n\n{w['snippet']}"
            for w in windows
        ]
    )

    evidence_records.append({
        "paper_id": row["paper_id"],
        "title": row["title"],
        "n_evidence_windows": len(windows),
        "evidence_text": evidence_text
    })

evidence_df = pd.DataFrame(evidence_records)

evidence_df[
    ["paper_id", "title", "n_evidence_windows"]
]


,paper_id,title,n_evidence_windows
0,2025.emnlp-main.86,Beyond the Surface: Measuring Self-Preference ...,15
1,2025.emnlp-main.103,Direct Judgement Preference Optimization,15
2,2025.emnlp-main.138,From Generation to Judgment: Opportunities and...,15
3,2025.emnlp-main.360,Arena-lite: Efficient and Reliable Large Langu...,15
4,2025.emnlp-main.538,Are Checklists Really Useful for Automatic Eva...,15
5,2025.emnlp-main.569,Analyzing Uncertainty of LLM -as-a-Judge: Inte...,15
6,2025.emnlp-main.583,Meta-Rewarding Language Models: Self-Improving...,15
7,2025.emnlp-main.625,Self-Augmented Preference Alignment for Sycoph...,15
8,2025.emnlp-main.712,Improve LLM -as-a-Judge Ability as a General A...,15
9,2025.emnlp-main.716,Towards Optimal Evaluation Efficiency for Larg...,8


## 7. Save evidence windows

These evidence snippets are useful for manual checking and make the later LLM output easier to verify.


In [8]:
EVIDENCE_FILE = Path(
    "../data/processed/cluster7_fulltext_evidence.csv"
)

evidence_df.to_csv(
    EVIDENCE_FILE,
    index=False
)

print("Saved:", EVIDENCE_FILE)


Saved: ../data/processed/cluster7_fulltext_evidence.csv


## 8. Set up OpenAI-assisted full-text validation

The model receives:

- paper title
- abstract
- relevant full-text evidence windows

It must distinguish between:

- `Yes`: directly supported by full-text evidence
- `No`: full text clearly indicates it was not studied
- `Partial`: some aspect is studied, but incompletely
- `Unclear`: available evidence is insufficient

This is stricter than the abstract-level screen.


In [9]:
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found. Check your .env file."
    )

client = OpenAI(api_key=api_key)

MODEL = "gpt-5.6-luna"

print("OpenAI client ready.")


OpenAI client ready.


## 9. Full-text validation function


In [12]:
FULLTEXT_FIELDS = [
    "multilingual_tested",
    "languages_tested",
    "low_resource_languages",
    "robustness_tested",
    "robustness_types",
    "prompt_variation_tested",
    "judge_bias_tested",
    "human_comparison",
    "judge_models",
    "explicit_limitations",
    "explicit_future_work",
    "supports_multilingual_gap",
    "supports_robustness_gap",
    "evidence_summary"
]


def validate_fulltext(
    title,
    abstract,
    evidence_text
):
    prompt = f"""
You are validating possible research gaps in the LLM-as-a-Judge literature.

Use ONLY the paper information supplied below.

TITLE:
{title}

ABSTRACT:
{abstract}

RELEVANT FULL-TEXT EVIDENCE:
{evidence_text}

Candidate gap A:
Multilingual reliability of LLM-as-a-Judge.

Candidate gap B:
Robustness / stability of LLM-as-a-Judge, including sensitivity to prompt wording, evaluation setup, repeated runs, order, domain, model, or other perturbations.

Return ONLY valid JSON with exactly these keys:
{json.dumps(FULLTEXT_FIELDS)}

Coding rules:

For:
- multilingual_tested
- low_resource_languages
- robustness_tested
- prompt_variation_tested
- judge_bias_tested
- human_comparison
- supports_multilingual_gap
- supports_robustness_gap

Use only:
Yes / No / Partial / Unclear

Definitions:
- Yes = directly supported by supplied evidence
- No = supplied evidence clearly shows the dimension was not studied or explicitly states its absence
- Partial = a related but incomplete version was studied
- Unclear = evidence is insufficient

Additional rules:
- languages_tested: list explicitly named languages, otherwise "Unclear"
- robustness_types: list only robustness dimensions explicitly tested
- judge_models: list only models explicitly named as judges/evaluators
- explicit_limitations: quote or tightly paraphrase only limitations evidenced in supplied text; otherwise "Not established from supplied evidence"
- explicit_future_work: only explicit future-work directions; otherwise "Not established from supplied evidence"
- supports_multilingual_gap: judge whether THIS PAPER provides evidence consistent with the multilingual gap candidate; do not make a corpus-level claim
- supports_robustness_gap: same rule for robustness
- evidence_summary: concise explanation of the strongest relevant evidence

Do not infer that something was absent merely because it was not found in the supplied snippets.
"""

    response = client.responses.create(
        model=MODEL,
        input=prompt
    )

    text = response.output_text.strip()

    if text.startswith("```"):
        text = text.strip("`")

        if text.lower().startswith("json"):
            text = text[4:].strip()

    return json.loads(text)


## 10. Test full-text validation on one paper first


In [13]:
test_row = evidence_df.iloc[0]

paper_row = papers[
    papers["paper_id"] == test_row["paper_id"]
].iloc[0]

test_result = validate_fulltext(
    title=paper_row["title"],
    abstract=paper_row["abstract"],
    evidence_text=test_row["evidence_text"]
)

print("Paper:", paper_row["title"])
print(
    json.dumps(
        test_result,
        indent=2,
        ensure_ascii=False
    )
)


Paper: Beyond the Surface: Measuring Self-Preference in LLM Judgments
{
  "multilingual_tested": "Partial",
  "languages_tested": [
    "German-English (de-en)"
  ],
  "low_resource_languages": "Unclear",
  "robustness_tested": "Partial",
  "robustness_types": [
    "model version",
    "model size",
    "reasoning ability",
    "response text style",
    "post-training data",
    "response order/position bias mitigation"
  ],
  "prompt_variation_tested": "Unclear",
  "judge_bias_tested": "Yes",
  "human_comparison": "Yes",
  "judge_models": [
    "GPT-4o-mini",
    "Gemini-1.5-Flash",
    "DeepSeek-V3",
    "Llama-3.1-8B",
    "Llama-3.1-8B-Instruct",
    "Llama-3.1-70B",
    "Llama-3.1-70B-Instruct",
    "Qwen2.5-0.5B-Instruct",
    "Qwen2.5-1.5B-Instruct",
    "Qwen2.5-3B-Instruct",
    "Qwen2.5-7B",
    "Qwen2.5-7B-Instruct",
    "Qwen2.5-14B-Instruct",
    "Qwen2.5-32B-Instruct",
    "Qwen2.5-72B",
    "Qwen2.5-72B-Instruct",
    "gemma-2-9B",
    "gemma-2-9B-it"
  ],
  "explicit_

## 11. Run validation across Cluster 7

Progress is saved after every paper.

If the script stops, rerun this cell and completed rows will be skipped.


In [14]:
FULLTEXT_REVIEW_FILE = Path(
    "../data/processed/cluster7_fulltext_validation.csv"
)

base_columns = [
    "paper_id",
    "title"
] + FULLTEXT_FIELDS

if FULLTEXT_REVIEW_FILE.exists():
    fulltext_review = pd.read_csv(
        FULLTEXT_REVIEW_FILE
    ).fillna("")
    print("Resuming:", FULLTEXT_REVIEW_FILE)

else:
    fulltext_review = papers[
        ["paper_id", "title"]
    ].copy()

    for field in FULLTEXT_FIELDS:
        fulltext_review[field] = ""


for i, paper in papers.iterrows():

    paper_id = paper["paper_id"]

    existing = fulltext_review.loc[
        fulltext_review["paper_id"] == paper_id,
        "evidence_summary"
    ]

    if len(existing) and str(existing.iloc[0]).strip():
        print(
            f"Skipping {i+1}/{len(papers)}: "
            "already processed"
        )
        continue

    evidence_match = evidence_df[
        evidence_df["paper_id"] == paper_id
    ]

    if evidence_match.empty:
        print(
            f"Skipping {i+1}/{len(papers)}: "
            "no extracted evidence"
        )
        continue

    evidence_text = (
        evidence_match.iloc[0]["evidence_text"]
    )

    print(
        f"Processing {i+1}/{len(papers)}: "
        f"{paper['title']}"
    )

    try:
        result = validate_fulltext(
            title=paper["title"],
            abstract=paper["abstract"],
            evidence_text=evidence_text
        )

        mask = (
            fulltext_review["paper_id"]
            == paper_id
        )

        for field in FULLTEXT_FIELDS:
            value = result.get(
                field,
                "Unclear"
            )

            if isinstance(value, list):
                value = "; ".join(
                    map(str, value)
                )

            fulltext_review.loc[
                mask,
                field
            ] = value

        fulltext_review.to_csv(
            FULLTEXT_REVIEW_FILE,
            index=False
        )

        time.sleep(0.5)

    except Exception as e:
        print("Error:", e)

        fulltext_review.to_csv(
            FULLTEXT_REVIEW_FILE,
            index=False
        )


print("\nSaved:")
print(FULLTEXT_REVIEW_FILE)


Processing 1/18: Beyond the Surface: Measuring Self-Preference in LLM Judgments
Processing 2/18: Direct Judgement Preference Optimization
Processing 3/18: From Generation to Judgment: Opportunities and Challenges of LLM -as-a-judge
Processing 4/18: Arena-lite: Efficient and Reliable Large Language Model Evaluation via Tournament-Based Direct Comparisons
Processing 5/18: Are Checklists Really Useful for Automatic Evaluation of Generative Tasks?
Processing 6/18: Analyzing Uncertainty of LLM -as-a-Judge: Interval Evaluations with Conformal Prediction
Processing 7/18: Meta-Rewarding Language Models: Self-Improving Alignment with LLM -as-a-Meta-Judge
Processing 8/18: Self-Augmented Preference Alignment for Sycophancy Reduction in LLM s
Processing 9/18: Improve LLM -as-a-Judge Ability as a General Ability
Processing 10/18: Towards Optimal Evaluation Efficiency for Large Language Models
Processing 11/18: C heck E val: A reliable LLM -as-a-Judge framework for evaluating text generation using c

## 12. Review full-text validation results


In [15]:
fulltext_review = pd.read_csv(
    FULLTEXT_REVIEW_FILE
).fillna("")

fulltext_review[
    [
        "paper_id",
        "title",
        "multilingual_tested",
        "low_resource_languages",
        "robustness_tested",
        "prompt_variation_tested",
        "judge_bias_tested",
        "human_comparison"
    ]
]


,paper_id,title,multilingual_tested,low_resource_languages,robustness_tested,prompt_variation_tested,judge_bias_tested,human_comparison
0,2025.emnlp-main.86,Beyond the Surface: Measuring Self-Preference ...,Partial,Unclear,Partial,Unclear,Yes,Yes
1,2025.emnlp-main.103,Direct Judgement Preference Optimization,No,No,Yes,Yes,Yes,Yes
2,2025.emnlp-main.138,From Generation to Judgment: Opportunities and...,Partial,Unclear,Yes,Partial,Yes,Yes
3,2025.emnlp-main.360,Arena-lite: Efficient and Reliable Large Langu...,No,No,Yes,No,Partial,Partial
4,2025.emnlp-main.538,Are Checklists Really Useful for Automatic Eva...,No,No,Partial,Partial,Yes,Yes
5,2025.emnlp-main.569,Analyzing Uncertainty of LLM -as-a-Judge: Inte...,Unclear,Unclear,Partial,Yes,Yes,Yes
6,2025.emnlp-main.583,Meta-Rewarding Language Models: Self-Improving...,Unclear,Unclear,Partial,Partial,Yes,Yes
7,2025.emnlp-main.625,Self-Augmented Preference Alignment for Sycoph...,Unclear,Unclear,Partial,Yes,No,Unclear
8,2025.emnlp-main.712,Improve LLM -as-a-Judge Ability as a General A...,Yes,Unclear,Partial,Yes,Yes,Unclear
9,2025.emnlp-main.716,Towards Optimal Evaluation Efficiency for Larg...,Unclear,Unclear,Partial,Unclear,Unclear,Unclear


## 13. Summarise candidate-gap evidence


In [16]:
dimensions = [
    "multilingual_tested",
    "low_resource_languages",
    "robustness_tested",
    "prompt_variation_tested",
    "judge_bias_tested",
    "human_comparison",
    "supports_multilingual_gap",
    "supports_robustness_gap"
]

summary_rows = []

for col in dimensions:
    counts = (
        fulltext_review[col]
        .replace("", "Missing")
        .value_counts()
    )

    print(f"\n=== {col} ===")
    print(counts)

    summary_rows.append({
        "dimension": col,
        "Yes": int(counts.get("Yes", 0)),
        "No": int(counts.get("No", 0)),
        "Partial": int(counts.get("Partial", 0)),
        "Unclear": int(counts.get("Unclear", 0)),
        "Missing": int(counts.get("Missing", 0))
    })

fulltext_summary = pd.DataFrame(
    summary_rows
)

fulltext_summary



=== multilingual_tested ===
multilingual_tested
Unclear    8
No         5
Partial    3
Yes        2
Name: count, dtype: int64

=== low_resource_languages ===
low_resource_languages
Unclear    12
No          6
Name: count, dtype: int64

=== robustness_tested ===
robustness_tested
Partial    12
Yes         6
Name: count, dtype: int64

=== prompt_variation_tested ===
prompt_variation_tested
Yes        5
Partial    5
Unclear    4
No         4
Name: count, dtype: int64

=== judge_bias_tested ===
judge_bias_tested
Yes        10
Partial     3
Unclear     3
No          2
Name: count, dtype: int64

=== human_comparison ===
human_comparison
Yes        12
Unclear     5
Partial     1
Name: count, dtype: int64

=== supports_multilingual_gap ===
supports_multilingual_gap
Unclear    7
Partial    6
Yes        5
Name: count, dtype: int64

=== supports_robustness_gap ===
supports_robustness_gap
Partial    13
Yes         5
Name: count, dtype: int64


,dimension,Yes,No,Partial,Unclear,Missing
0,multilingual_tested,2,5,3,8,0
1,low_resource_languages,0,6,0,12,0
2,robustness_tested,6,0,12,0,0
3,prompt_variation_tested,5,4,5,4,0
4,judge_bias_tested,10,2,3,3,0
5,human_comparison,12,0,1,5,0
6,supports_multilingual_gap,5,0,6,7,0
7,supports_robustness_gap,5,0,13,0,0


## 14. Papers most relevant to each candidate gap


In [17]:
multilingual_candidates = fulltext_review[
    fulltext_review["supports_multilingual_gap"]
    .isin(["Yes", "Partial"])
][
    [
        "paper_id",
        "title",
        "multilingual_tested",
        "languages_tested",
        "low_resource_languages",
        "explicit_limitations",
        "explicit_future_work",
        "evidence_summary"
    ]
]

multilingual_candidates


,paper_id,title,multilingual_tested,languages_tested,low_resource_languages,explicit_limitations,explicit_future_work,evidence_summary
0,2025.emnlp-main.86,Beyond the Surface: Measuring Self-Preference ...,Partial,de-en,Unclear,The paper states that existing self-preference...,Not established from supplied evidence,The study includes WMT19 German-English (de-en...
1,2025.emnlp-main.103,Direct Judgement Preference Optimization,No,English,No,The paper focuses on evaluation in English.; T...,Determine how to build judges for multilingual...,The paper explicitly states that it focuses on...
2,2025.emnlp-main.138,From Generation to Judgment: Opportunities and...,Partial,Unclear,Unclear,"The survey identifies bias and vulnerability, ...",Future work should analyze the root causes of ...,The survey mentions multilingual generation an...
3,2025.emnlp-main.360,Arena-lite: Efficient and Reliable Large Langu...,No,English,No,"Randomness remains in tournament outcomes, inc...","Extend Arena-Lite to diverse domains, includin...",The study uses the English category of Chatbot...
4,2025.emnlp-main.538,Are Checklists Really Useful for Automatic Eva...,No,English,No,Pairwise comparison has ambiguity in evaluatio...,Not established from supplied evidence,The study evaluates English-only datasets: LLM...
5,2025.emnlp-main.569,Analyzing Uncertainty of LLM -as-a-Judge: Inte...,Unclear,Unclear,Unclear,The study primarily covers summarization and r...,Not established from supplied evidence,The paper does not name or evaluate any langua...
8,2025.emnlp-main.712,Improve LLM -as-a-Judge Ability as a General A...,Yes,English; Chinese,Unclear,The paper states that it cannot achieve consis...,The authors plan to study the interplay betwee...,The paper evaluates the judge under official E...
11,2025.emnlp-main.891,Multi-perspective Analysis of Large Language M...,Partial,Japanese,No,The study focuses narrowly on accounting audit...,Conduct experiments with different model sizes...,The paper uses Japanese-domain data and a mult...
13,2025.emnlp-main.992,Beyond the Score: Uncertainty-Calibrated LLM s...,No,English,No,The study is confined to English and three pub...,The authors plan to study model-size scaling f...,The paper explicitly limits evaluation to Engl...
15,2025.emnlp-main.1307,Co-Eval: Augmenting LLM -based Evaluation with...,Yes,English; Chinese; Thai; Spanish; French; Ukrai...,Unclear,LLM-based evaluations can suffer from bias and...,Not established from supplied evidence,The paper evaluates translation judgments acro...


In [18]:
robustness_candidates = fulltext_review[
    fulltext_review["supports_robustness_gap"]
    .isin(["Yes", "Partial"])
][
    [
        "paper_id",
        "title",
        "robustness_tested",
        "robustness_types",
        "prompt_variation_tested",
        "judge_bias_tested",
        "explicit_limitations",
        "explicit_future_work",
        "evidence_summary"
    ]
]

robustness_candidates


,paper_id,title,robustness_tested,robustness_types,prompt_variation_tested,judge_bias_tested,explicit_limitations,explicit_future_work,evidence_summary
0,2025.emnlp-main.86,Beyond the Surface: Measuring Self-Preference ...,Partial,model version; model size; reasoning ability; ...,Unclear,Yes,The paper states that existing self-preference...,Not established from supplied evidence,The study includes WMT19 German-English (de-en...
1,2025.emnlp-main.103,Direct Judgement Preference Optimization,Yes,positional/order sensitivity; prompt variation...,Yes,Yes,The paper focuses on evaluation in English.; T...,Determine how to build judges for multilingual...,The paper explicitly states that it focuses on...
2,2025.emnlp-main.138,From Generation to Judgment: Opportunities and...,Yes,position/order sensitivity; adversarial robust...,Partial,Yes,"The survey identifies bias and vulnerability, ...",Future work should analyze the root causes of ...,The survey mentions multilingual generation an...
3,2025.emnlp-main.360,Arena-lite: Efficient and Reliable Large Langu...,Yes,benchmark dataset size; judge accuracy; number...,No,Partial,"Randomness remains in tournament outcomes, inc...","Extend Arena-Lite to diverse domains, includin...",The study uses the English category of Chatbot...
4,2025.emnlp-main.538,Are Checklists Really Useful for Automatic Eva...,Partial,repeated evaluation runs; response order/posit...,Partial,Yes,Pairwise comparison has ambiguity in evaluatio...,Not established from supplied evidence,The study evaluates English-only datasets: LLM...
5,2025.emnlp-main.569,Analyzing Uncertainty of LLM -as-a-Judge: Inte...,Partial,LLM judge model choice; prompting strategy; ju...,Yes,Yes,The study primarily covers summarization and r...,Not established from supplied evidence,The paper does not name or evaluate any langua...
6,2025.emnlp-main.583,Meta-Rewarding Language Models: Self-Improving...,Partial,positional/order bias; length bias; score bias,Partial,Yes,The paper identifies rapid saturation and poss...,Not established from supplied evidence,The study does not explicitly name or evaluate...
7,2025.emnlp-main.625,Self-Augmented Preference Alignment for Sycoph...,Partial,prompt template variation; task variation,Yes,No,Performance under the negated suggestion templ...,Not established from supplied evidence,The paper evaluates multiple open-source model...
8,2025.emnlp-main.712,Improve LLM -as-a-Judge Ability as a General A...,Partial,Prompt language variation; Prompt instruction-...,Yes,Yes,The paper states that it cannot achieve consis...,The authors plan to study the interplay betwee...,The paper evaluates the judge under official E...
9,2025.emnlp-main.716,Towards Optimal Evaluation Efficiency for Larg...,Partial,random-seed sensitivity; compression rate; sub...,Unclear,Unclear,The correlation between semantic retention and...,Refine the optimized function by exploring dif...,The paper studies efficient benchmark subset s...


In [ ]:
The main finding is that the multilingual direction looks much stronger as a gap candidate than the generic robustness direction.
From the full-text validation:
- Multilingual tested: 2 Yes, 3 Partial, 5 No, 8 Unclear
- Low-resource languages: 0 Yes, 0 Partial, 6 No, 12 Unclear
- Robustness tested: 6 Yes, 12 Partial
- Prompt variation tested: 5 Yes, 5 Partial, 4 No, 4 Unclear
- Judge bias tested: 10 Yes, 3 Partial
- Human comparison: 12 Yes, 1 Partial
That means robustness is actually already quite heavily studied in this cluster. Every paper had at least some robustness-related evaluation, and many explicitly tested prompt variation, model changes, bias, ordering, repeated runs, or related perturbations.
By contrast, low-resource multilingual judging is barely represented. None of the 18 papers were coded as clearly testing low-resource languages.
There is also a particularly useful piece of evidence in Direct Judgement Preference Optimization: its extracted future-work text explicitly points toward building judges for multilingual settings. Other papers are English-only or limited to a small number of languages, while one paper does cover several languages but does not appear to specifically investigate low-resource-language reliability.
So I would now narrow your candidate gap from:
multilingual LLM-as-a-Judge

to something more specific, such as:
Reliability and consistency of LLM-as-a-Judge in low-resource languages

or potentially:
Cross-lingual consistency of LLM-as-a-Judge between high-resource and low-resource languages

A possible research question would be:
How consistent are LLM-as-a-Judge evaluations across high-resource and low-resource languages, and how closely do these judgments align with human evaluators?

That is much stronger because your own literature-mining pipeline has now produced evidence for why you are investigating it.